# Klasifikasi Mobil vs Motor dengan MobileNetV3-Large (Google Colab)

**Cara pakai**
1. Menu **Runtime → Change runtime type → T4 GPU**.
2. Menu **Runtime → Run all**.
3. Saat sel upload muncul, pilih file **dataset_patrol.zip**.

**Alur:** upload zip → bersihkan data (flatten, hapus duplikat & file rusak) → split train/val/test → latih head → fine-tuning → evaluasi → simpan model → coba prediksi gambar baru.

**Catatan penting:** input MobileNetV3 di Keras harus piksel **0–255** (preprocessing sudah ada di dalam model). Jangan menambahkan `Rescaling(1/255)`.

## 1. Setup & konfigurasi

In [ ]:
import os, io, json, shutil, zipfile, hashlib
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV3Large

SEED = 42
tf.keras.utils.set_random_seed(SEED)

# --- Konfigurasi (ubah di sini bila perlu) ---
CLASSES = ['car', 'motorcycle']   # nama folder kelas di dalam zip
IMG = 224                         # ukuran input model
BATCH = 16
EPOCHS_HEAD = 30                  # tahap 1: hanya melatih head
EPOCHS_FINE = 40                  # tahap 2: fine-tuning
LR_HEAD = 1e-3
LR_FINE = 2e-5
UNFREEZE_LAST = 30                # jumlah layer terakhir base model yang dilatih saat fine-tuning
RUN_CV = False                    # True = jalankan 5-fold cross-validation (lebih tepercaya untuk data kecil)

ZIP_PATH = '/content/dataset_patrol.zip'
RAW_DIR = '/content/data/raw'
CLEAN_DIR = '/content/data/clean'
OUT_DIR = '/content/output'
os.makedirs(OUT_DIR, exist_ok=True)

print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus if gpus else 'TIDAK ADA -> aktifkan lewat Runtime > Change runtime type > T4 GPU')

## 2. Upload dataset_patrol.zip

In [ ]:
from google.colab import files

if not os.path.exists(ZIP_PATH):
    uploaded = files.upload()
    zips = [n for n in uploaded if n.lower().endswith('.zip')]
    assert zips, 'Tidak ada file .zip yang diupload'
    shutil.move(zips[0], ZIP_PATH)

# Alternatif lewat Google Drive (hapus tanda # lalu sesuaikan path):
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy('/content/drive/MyDrive/dataset_patrol.zip', ZIP_PATH)

print('ZIP siap:', ZIP_PATH, f'({os.path.getsize(ZIP_PATH) / 1e6:.1f} MB)')

## 3. Bersihkan data
Ekstrak zip, ratakan folder bersarang, buang file duplikat/rusak/bukan gambar, perbaiki rotasi EXIF, dan kecilkan gambar yang sangat besar (maks. 1024 px) agar training cepat.

In [ ]:
EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.gif')

shutil.rmtree(RAW_DIR, ignore_errors=True)
shutil.rmtree(CLEAN_DIR, ignore_errors=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(RAW_DIR)
for c in CLASSES:
    os.makedirs(f'{CLEAN_DIR}/{c}')

seen = set()
count = {c: 0 for c in CLASSES}
skipped = {'bukan gambar': 0, 'kelas tidak dikenal': 0, 'duplikat': 0, 'rusak': 0}

for root, _, fnames in os.walk(RAW_DIR):
    for fn in sorted(fnames):
        path = os.path.join(root, fn)
        if '__MACOSX' in path or fn.startswith('.') or not fn.lower().endswith(EXTS):
            skipped['bukan gambar'] += 1
            continue
        folders = [p.lower() for p in os.path.relpath(path, RAW_DIR).split(os.sep)[:-1]]
        cls = next((c for c in CLASSES if c in folders), None)
        if cls is None:
            skipped['kelas tidak dikenal'] += 1
            continue
        with open(path, 'rb') as f:
            data = f.read()
        h = hashlib.md5(data).hexdigest()
        if h in seen:
            skipped['duplikat'] += 1
            continue
        try:
            im = ImageOps.exif_transpose(Image.open(io.BytesIO(data))).convert('RGB')
        except Exception:
            skipped['rusak'] += 1
            continue
        seen.add(h)
        im.thumbnail((1024, 1024))
        count[cls] += 1
        im.save(f'{CLEAN_DIR}/{cls}/{cls}_{count[cls]:04d}.jpg', quality=95)

print('Gambar bersih per kelas:', count)
print('Dilewati:', skipped)
assert all(v > 0 for v in count.values()), 'Ada kelas tanpa gambar. Cek nama folder di dalam zip (harus memuat "car" dan "motorcycle").'

## 4. Lihat contoh gambar & bagi data (70% train / 15% val / 15% test)

In [ ]:
def list_files(d):
    paths, labels = [], []
    for i, c in enumerate(CLASSES):
        for fn in sorted(os.listdir(f'{d}/{c}')):
            paths.append(f'{d}/{c}/{fn}')
            labels.append(i)
    return np.array(paths), np.array(labels)

paths, labels = list_files(CLEAN_DIR)

# split stratified: proporsi kelas sama di train/val/test
p_tr, p_tmp, y_tr, y_tmp = train_test_split(paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
p_va, p_te, y_va, y_te = train_test_split(p_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
for name, y in [('train', y_tr), ('val', y_va), ('test', y_te)]:
    print(f'{name:5s}: {len(y):3d} gambar ->', {c: int((y == i).sum()) for i, c in enumerate(CLASSES)})

# contoh gambar
rng = np.random.RandomState(SEED)
idx = rng.choice(len(paths), 8, replace=False)
plt.figure(figsize=(14, 4))
for k, i in enumerate(idx):
    plt.subplot(2, 4, k + 1)
    plt.imshow(Image.open(paths[i]))
    plt.title(CLASSES[labels[i]])
    plt.axis('off')
plt.tight_layout()
plt.show()

## 5. Pipeline data
Gambar di-resize **dengan padding** (rasio tidak penyok) dan nilainya tetap **0–255**.

In [ ]:
def load_image(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize_with_pad(img, IMG, IMG)
    img.set_shape((IMG, IMG, 3))
    return tf.cast(img, tf.float32)          # 0..255, JANGAN dibagi 255

def make_ds(p, y, train):
    ds = tf.data.Dataset.from_tensor_slices((p, y))
    ds = ds.map(lambda path, label: (load_image(path), label),
                num_parallel_calls=tf.data.AUTOTUNE).cache()
    if train:
        ds = ds.shuffle(len(p), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

## 6. Model: MobileNetV3-Large (transfer learning)

In [ ]:
def build_model():
    augment = tf.keras.Sequential([
        layers.RandomFlip('horizontal'),
        layers.RandomRotation(0.15),
        layers.RandomZoom((-0.2, 0.2)),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomContrast(0.2),
        layers.RandomBrightness(0.2, value_range=(0, 255)),
    ], name='augmentation')

    base = MobileNetV3Large(input_shape=(IMG, IMG, 3), include_top=False,
                            weights='imagenet', include_preprocessing=True)
    base.trainable = False

    inp = tf.keras.Input((IMG, IMG, 3))
    x = augment(inp)                       # augmentasi hanya aktif saat training
    x = base(x, training=False)            # BatchNorm base tetap mode inference
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(len(CLASSES), activation='softmax')(x)
    return models.Model(inp, out), base

def compile_model(model, lr):
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                  metrics=['accuracy'])

def make_callbacks():
    return [
        tf.keras.callbacks.EarlyStopping('val_loss', patience=8, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.3, patience=3, min_lr=1e-7),
    ]

def train_model(p_train, y_train, p_val, y_val, verbose=1):
    """Latih 2 tahap: (1) head saja, (2) fine-tuning layer terakhir. Mengembalikan model + riwayat."""
    train_ds = make_ds(p_train, y_train, True)
    val_ds = make_ds(p_val, y_val, False)
    cw = {i: len(y_train) / (len(CLASSES) * n) for i, n in enumerate(np.bincount(y_train, minlength=len(CLASSES)))}

    model, base = build_model()

    # Tahap 1: base dibekukan, latih head
    compile_model(model, LR_HEAD)
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
                   class_weight=cw, callbacks=make_callbacks(), verbose=verbose)

    # Tahap 2: buka N layer terakhir, BatchNorm tetap beku, learning rate kecil
    base.trainable = True
    for l in base.layers[:-UNFREEZE_LAST]:
        l.trainable = False
    for l in base.layers:
        if isinstance(l, layers.BatchNormalization):
            l.trainable = False
    compile_model(model, LR_FINE)
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINE,
                   class_weight=cw, callbacks=make_callbacks(), verbose=verbose)
    return model, h1, h2

## 7. Training

In [ ]:
model, h1, h2 = train_model(p_tr, y_tr, p_va, y_va)

## 8. Kurva belajar

In [ ]:
def cat(key):
    return h1.history[key] + h2.history[key]

start_ft = len(h1.history['loss'])
plt.figure(figsize=(12, 4))
for k, (title, a, b) in enumerate([('Accuracy', 'accuracy', 'val_accuracy'), ('Loss', 'loss', 'val_loss')]):
    plt.subplot(1, 2, k + 1)
    plt.plot(cat(a), label='Train')
    plt.plot(cat(b), label='Val')
    plt.axvline(start_ft - 0.5, color='gray', linestyle='--', label='mulai fine-tuning')
    plt.title(title)
    plt.xlabel('epoch')
    plt.legend()
plt.tight_layout()
plt.show()

## 9. Evaluasi di test set (data yang tidak pernah dilihat saat training)

In [ ]:
test_ds = make_ds(p_te, y_te, False)
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
probs = model.predict(test_ds, verbose=0)
y_pred = probs.argmax(axis=1)

print(f'Test accuracy: {test_acc * 100:.1f}%  ({int((y_pred == y_te).sum())}/{len(y_te)} benar)   Test loss: {test_loss:.4f}')
print(classification_report(y_te, y_pred, target_names=CLASSES, zero_division=0))

ConfusionMatrixDisplay(confusion_matrix(y_te, y_pred, labels=range(len(CLASSES))),
                       display_labels=CLASSES).plot(cmap='Blues', values_format='d')
plt.title('Confusion matrix (test set)')
plt.show()

# gambar yang salah ditebak
wrong = np.where(y_pred != y_te)[0]
if len(wrong):
    plt.figure(figsize=(14, 3.5))
    for k, i in enumerate(wrong[:8]):
        plt.subplot(1, min(len(wrong), 8), k + 1)
        plt.imshow(Image.open(p_te[i]))
        plt.title(f'asli: {CLASSES[y_te[i]]}\ntebak: {CLASSES[y_pred[i]]} ({probs[i].max() * 100:.0f}%)', fontsize=9)
        plt.axis('off')
    plt.show()
else:
    print('Tidak ada gambar yang salah di test set.')

print('\nCATATAN: test set hanya', len(y_te), 'gambar, jadi satu kesalahan = ~', round(100 / len(y_te), 1),
      'poin akurasi. Untuk angka yang lebih tepercaya jalankan sel cross-validation di bawah (RUN_CV = True).')

## 10. (Opsional) 5-fold cross-validation
Set `RUN_CV = True` di sel konfigurasi, jalankan ulang sel itu, lalu jalankan sel ini. Setiap gambar dipakai sebagai data uji tepat satu kali, sehingga hasilnya jauh lebih stabil daripada satu kali split.

In [ ]:
if RUN_CV:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_acc = []
    for k, (tr, te) in enumerate(skf.split(paths, labels), 1):
        tf.keras.backend.clear_session()
        a_p, v_p, a_y, v_y = train_test_split(paths[tr], labels[tr], test_size=0.15,
                                              stratify=labels[tr], random_state=SEED)
        m, _, _ = train_model(a_p, a_y, v_p, v_y, verbose=0)
        _, acc = m.evaluate(make_ds(paths[te], labels[te], False), verbose=0)
        fold_acc.append(acc)
        print(f'Fold {k}: akurasi {acc * 100:.1f}% ({len(te)} gambar uji)')
    print(f'\nRata-rata 5-fold: {np.mean(fold_acc) * 100:.1f}% ± {np.std(fold_acc) * 100:.1f}%')
    tf.keras.backend.clear_session()
else:
    print('Cross-validation dilewati (RUN_CV = False).')

## 11. Simpan model & unduh

In [ ]:
model.save(f'{OUT_DIR}/mobilenetv3_patrol.keras')
with open(f'{OUT_DIR}/classes.json', 'w') as f:
    json.dump(CLASSES, f)
print('Tersimpan di', OUT_DIR, os.listdir(OUT_DIR))

shutil.make_archive('/content/patrol_model', 'zip', OUT_DIR)
files.download('/content/patrol_model.zip')

# Simpan juga ke Google Drive (opsional, hapus tanda #):
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy('/content/patrol_model.zip', '/content/drive/MyDrive/patrol_model.zip')

## 12. (Opsional) Ekspor TFLite untuk Raspberry Pi / Jetson
Ukurannya jauh lebih kecil dan lebih cepat di perangkat edge. Pada TFLite, input tetap **piksel 0–255 (float32)**.

In [ ]:
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
    with open(f'{OUT_DIR}/mobilenetv3_patrol.tflite', 'wb') as f:
        f.write(tflite_model)
    print(f'TFLite tersimpan ({len(tflite_model) / 1e6:.1f} MB)')
    files.download(f'{OUT_DIR}/mobilenetv3_patrol.tflite')
except Exception as e:
    print('Ekspor TFLite gagal (model .keras tetap aman dipakai):', e)

## 13. Coba prediksi gambar baru
Upload satu atau beberapa foto (sebaiknya dari kamera robot: malam, hujan, sudut miring, tertutup sebagian).

In [ ]:
def predict_images(file_paths):
    for pth in file_paths:
        im = ImageOps.exif_transpose(Image.open(pth)).convert('RGB')
        x = tf.image.resize_with_pad(np.array(im), IMG, IMG)       # 0..255, sama seperti saat training
        p = model.predict(tf.expand_dims(x, 0), verbose=0)[0]
        plt.figure(figsize=(3.5, 3.5))
        plt.imshow(im)
        plt.axis('off')
        plt.title(f'{CLASSES[p.argmax()]} ({p.max() * 100:.1f}%)')
        plt.show()

new_up = files.upload()
predict_images(list(new_up.keys()))